# TP 2 : Automatiser le pipeline avec Prefect (CORRECTION)

**BUT INFO3 Développeur IA, R5.C.06, DATA5, Semaine 45**
M. Sakkari, IUT de La Rochelle

> **Version enseignant :** codes complétés et réponses aux questions.

## Dans ce TP, on apprend à...
- transformer un pipeline Python en flux Prefect ;
- découper un flux en tâches ;
- écrire un journal avec Prefect ;
- relancer automatiquement une tâche qui échoue ;
- arrêter un flux quand le fichier source est abîmé ;
- lancer un flux à intervalle régulier ;
- lancer un flux seulement quand un fichier a changé.

> **Comment travailler :**
> - Une cellule « Démo » est déjà écrite : lancez-la et lisez-la.
> - Une cellule « À vous » est à compléter : remplacez les `...` par votre code.
> - Comparez votre résultat avec le « Résultat attendu » écrit dans la cellule.

## Le cas
Vous reprenez le pipeline du Café Rochelais (TP S42). Il est déjà écrit. Décompressez le zip `TP2_Prefect_etudiants.zip` : le dossier obtenu, `TP2_Prefect`, est le **dossier du TP**. Il contient ce notebook et ces fichiers :

| Fichier | Contenu |
|---|---|
| `etl.py` | le pipeline du TP S42 |
| `outils.py` | quelques outils : contrôle d'un fichier, source en panne, nouvelle commande, empreinte |
| `data/` | les quatre fichiers de données |

## Objectif
Faire tourner ce pipeline tout seul, de façon fiable : le relancer en cas de panne, garder une trace, et le lancer au bon moment.

## Avant de commencer
1. Ouvrez le dossier du TP dans VS Code.
2. Ouvrez un terminal dans le dossier du TP. Créez un environnement : `python -m venv env`.
3. Activez-le : `env\Scripts\activate` (Windows) ou `source env/bin/activate` (Linux, macOS).
4. Installez les paquets : `pip install prefect pandas ipykernel`.
5. Dans VS Code, choisissez l'environnement `env` comme noyau du notebook (en haut à droite).
6. Exécutez la cellule ci-dessous.

In [24]:
# Cette cellule vérifie l'installation. Il n'y a rien à compléter.
import json
import os
import sqlite3
from collections import Counter

# Au premier lancement, Prefect démarre un serveur temporaire : on lui laisse le temps (en secondes)
os.environ["PREFECT_SERVER_EPHEMERAL_STARTUP_TIMEOUT_SECONDS"] = "120"

import pandas as pd
import prefect
from prefect import flow, get_run_logger, task
from prefect.runtime import flow_run

import etl       # le pipeline du TP S42, déjà écrit
import outils    # quelques outils fournis

# Windows affiche parfois une erreur sans gravité quand une connexion se ferme : on la cache
import logging
logging.getLogger("asyncio").setLevel(logging.CRITICAL)


# Au premier lancement, Prefect démarre un serveur temporaire. On le démarre ici, une seule fois.
# Les messages de cette cellule sont normaux.
@flow
def demarrage():
    return "ok"


for essai in range(3):
    try:
        demarrage()
        break
    except Exception:
        pass

print("Prefect", prefect.__version__, "est installé.")

12:23:57.186 | INFO    | Flow run 'gregarious-alpaca' - Beginning flow run 'gregarious-alpaca' for flow 'demarrage'

12:23:58.190 | INFO    | Flow run 'gregarious-alpaca' - Finished in state Completed()

Prefect 3.9.1 est installé.


## Petit lexique avant de commencer

| Mot | Ce que ça veut dire | Exemple |
|---|---|---|
| **Tâche** (*task*) | la plus petite unité de travail d'un flux : elle fait une seule opération | charger les lignes dans la base |
| **Flux** (*flow*) | un enchaînement de tâches qui forme un traitement complet | extraire, transformer, charger |
| **Déclencheur** | l'événement dont l'arrivée lance un flux | une heure, un fichier qui change |
| **Planification** | fixer à l'avance les instants où un flux s'exécute | toutes les 30 secondes |
| **Relance** (*retry*) | refaire automatiquement une tâche qui a échoué, un nombre limité de fois | une source en panne passagère |
| **Orchestrateur** | l'outil qui coordonne les flux : il les lance, enchaîne les tâches, relance, et garde la trace | Prefect |

**Définitions détaillées :** support de cours S45, diapositives « Vocabulaire » (5 à 10).

## Partie A : Un premier flux

### Démo : A.1 un flux avec une tâche
Cette cellule est donnée. Lisez-la, puis exécutez-la.

Dans Prefect, on écrit une fonction Python normale. Une ligne qui commence par `@`, placée juste au-dessus d'elle, la transforme :
- `@task` en fait une tâche ;
- `@flow` en fait un flux.

Cette ligne s'appelle un **décorateur**.

In [3]:
# Cette cellule définit une tâche et un flux. Il n'y a rien à compléter.

@task                          # @task transforme une fonction Python en tâche Prefect
def dire_bonjour(prenom):
    return "Bonjour " + prenom


@flow                          # @flow transforme une fonction en flux : un enchaînement de tâches
def accueil():
    return dire_bonjour("Niort")


# On lance le flux comme une fonction normale
print(accueil())

11:52:21.632 | INFO    | Flow run 'dainty-rooster' - Beginning flow run 'dainty-rooster' for flow 'accueil'

11:52:21.651 | INFO    | Task run 'dire_bonjour-402' - Finished in state Completed()

11:52:22.635 | INFO    | Flow run 'dainty-rooster' - Finished in state Completed()

Bonjour Niort


**Lecture :** Prefect affiche des messages pendant l'exécution.
- Au premier lancement, un message dit que Prefect démarre un serveur temporaire.
- Un message dit que le flux démarre.
- Un message dit que la tâche `dire_bonjour` a fini à l'état « Completed ».
- Un message dit que le flux a fini.

Le flux renvoie « Bonjour Niort », que `print` affiche.

### Tâche ou flux ?
Pour savoir quoi écrire, regardez ce que fait la fonction :
- Une fonction qui fait **une seule opération** devient une **tâche** (`@task`).
- Une fonction qui **enchaîne plusieurs tâches** devient un **flux** (`@flow`).
- Un flux peut aussi appeler un autre flux. On l'appelle alors un **sous-flux**.

**Schéma 1 : la démo A.1.** Le flux `accueil` appelle une seule tâche.
```
flux accueil
  └── tâche dire_bonjour  ──►  "Bonjour Niort"
```

**Schéma 2 : le pipeline de la partie B.** Le flux enchaîne trois tâches, dans cet ordre.
```
flux pipeline_etl
  tâche extraire  ──►  tâche transformer  ──►  tâche charger
```

**Schéma 3 : la partie E.** Le flux `pipeline_securise` appelle une tâche, puis le sous-flux `pipeline_etl`.
```
flux pipeline_securise
  tâche controler_source  ──►  sous-flux pipeline_etl
                                 tâche extraire  ──►  tâche transformer  ──►  tâche charger
```

Dans chaque schéma, une flèche veut dire : « la suite attend que l'étape précédente ait réussi ».

### À vous : A.2 votre premier flux
**À faire :**
1. Écrivez une tâche `additionner(a, b)` qui renvoie la somme de ses deux nombres.
2. Écrivez un flux `calcul()` qui appelle `additionner` avec 2 et 3, et renvoie le résultat.

Indice : les décorateurs `@task` et `@flow`, comme dans la démo.

In [4]:
# Objectif : écrire votre premier flux.
# Résultat attendu : le flux affiche 5.

# Étape 1 : la tâche additionner
@task
def additionner(a, b):
    return a + b

# Étape 2 : le flux calcul
@flow
def calcul():
    return additionner(2, 3)

print(calcul())

11:52:22.843 | INFO    | Flow run 'serious-spaniel' - Beginning flow run 'serious-spaniel' for flow 'calcul'

11:52:22.854 | INFO    | Task run 'additionner-614' - Finished in state Completed()

11:52:23.846 | INFO    | Flow run 'serious-spaniel' - Finished in state Completed()

5


**Lecture :** le flux renvoie 5. Dans les messages de Prefect, la tâche `additionner` finit à l'état « Completed »,
puis le flux `calcul` aussi.

## Partie B : Le pipeline en tâches

### Démo : B.1 repartir d'une base vide

In [5]:
# Cette cellule supprime la base cafe.db, pour repartir de zéro. Il n'y a rien à compléter.
if os.path.exists(etl.DB):
    os.remove(etl.DB)
print("La base", etl.DB, "n'existe plus. Elle sera recréée au premier chargement.")

La base cafe.db n'existe plus. Elle sera recréée au premier chargement.


### À vous : B.2 transformer le pipeline en flux
Le module `etl` contient déjà les fonctions du TP S42. Vous les enveloppez dans des tâches.

**À faire :** écrivez trois tâches et un flux.
1. Tâche `extraire(fichier)` : elle lit les trois fichiers avec `etl.extract(fichier)`, qui renvoie `clients, commandes, produits`.
2. Tâche `transformer(clients, commandes, produits)` :
   - `etl.transform_clients(clients)` renvoie `clients_ok, remap, rejets_clients` :
     les clients gardés, la correspondance « client en double → client gardé » (`remap`), et les clients rejetés ;
   - `etl.transform_commandes(commandes, clients_ok, remap, produits)` renvoie `commandes_ok, rejets_cmd, nb_dup` :
     les commandes gardées, les commandes rejetées, et le nombre de commandes en double supprimées (`nb_dup`) ;
   - la tâche renvoie `clients_ok, commandes_ok`, la liste de tous les rejets (`rejets_clients + rejets_cmd`) et `nb_dup`.
3. Tâche `charger(...)` : elle ouvre la base `etl.DB`, appelle `etl.load(...)`, puis ferme la base.
   - `etl.load(con, clients_ok, produits, commandes_ok, rejets, execution, fichier, lus, nb_dup, declencheur)` ;
   - `execution` est le nom de l'exécution Prefect : `flow_run.name` ;
   - `declencheur` vaut `"prefect"`.
4. Flux `pipeline_etl(fichier)` : il enchaîne les trois tâches. `lus` est le nombre de lignes de `commandes`.

In [6]:
# Objectif : transformer le pipeline en flux Prefect, avec trois tâches.
# Résultat attendu : à la fin du flux, la table commande contient 1879 lignes.

@task
def extraire(fichier):
    # Étape 1 : lire les trois fichiers et renvoyer les trois tableaux
    return etl.extract(fichier)


@task
def transformer(clients, commandes, produits):
    # Étape 1 : transformer les clients, puis les commandes
    # Étape 2 : renvoyer clients_ok, commandes_ok, tous les rejets, et nb_dup
    clients_ok, remap, rejets_clients = etl.transform_clients(clients)
    commandes_ok, rejets_cmd, nb_dup = etl.transform_commandes(commandes, clients_ok, remap, produits)
    return clients_ok, commandes_ok, rejets_clients + rejets_cmd, nb_dup


@task
def charger(clients_ok, produits, commandes_ok, rejets, fichier, lus, nb_dup):
    # Étape 1 : ouvrir la base, appeler etl.load, fermer la base
    con = sqlite3.connect(etl.DB)
    etl.load(con, clients_ok, produits, commandes_ok, rejets, flow_run.name, fichier, lus, nb_dup, "prefect")
    con.close()


@flow
def pipeline_etl(fichier="commandes.csv"):
    # Étape 1 : enchaîner les trois tâches, puis renvoyer clients_ok, commandes_ok et les rejets
    clients, commandes, produits = extraire(fichier)
    clients_ok, commandes_ok, rejets, nb_dup = transformer(clients, commandes, produits)
    charger(clients_ok, produits, commandes_ok, rejets, fichier, len(commandes), nb_dup)
    return clients_ok, commandes_ok, rejets


# Lancement du flux sur le fichier des commandes
pipeline_etl("commandes.csv")

# Contrôle : nombre de lignes de la table commande
con = sqlite3.connect(etl.DB)
print("La table commande contient", con.execute("SELECT COUNT(*) FROM commande").fetchone()[0], "lignes.")
con.close()

11:52:24.167 | INFO    | Flow run 'platinum-hedgehog' - Beginning flow run 'platinum-hedgehog' for flow 'pipeline-etl'

11:52:24.187 | INFO    | Task run 'extraire-f9a' - Finished in state Completed()

11:52:24.270 | INFO    | Task run 'transformer-788' - Finished in state Completed()

11:52:24.312 | INFO    | Task run 'charger-2fe' - Finished in state Completed()

11:52:25.172 | INFO    | Flow run 'platinum-hedgehog' - Finished in state Completed()

La table commande contient 1879 lignes.


**Lecture :** la base contient 1879 commandes, comme avec le pipeline du TP S42.
Dans les messages de Prefect, chaque tâche a son propre état : `extraire`, `transformer` et `charger` finissent chacune à « Completed ».

## Partie C : Le journal
### À vous : C.1 écrire dans le journal de Prefect
Prefect garde un journal de chaque exécution. On y écrit avec `get_run_logger()`.

**À faire :** écrivez la tâche `journaliser(lues, chargees, rejets)` et un flux `pipeline_etl_journal`.
1. Une **information** avec le nombre de commandes lues et chargées.
2. Un **avertissement** par motif de rejet, avec le nombre de lignes.
   Le motif d'un rejet est le 3e élément de son tuple. Les motifs `doublon_de_...` se regroupent sous `doublon_client`.
3. Le flux reprend les trois tâches de la partie B, puis appelle `journaliser`.

Indices : `journal.info(...)`, `journal.warning(...)`, et `Counter` pour compter.

In [7]:
# Objectif : écrire dans le journal de Prefect ce qui s'est passé.
# Résultat attendu : dans les messages du flux, une ligne d'information avec 2035 commandes lues et 1879 chargées, puis un avertissement par motif de rejet.

@task
def journaliser(lues, chargees, rejets):
    journal = get_run_logger()
    # Étape 1 : une information avec le nombre de commandes lues et chargées
    # Étape 2 : un avertissement par motif de rejet, avec le nombre de lignes
    journal.info("%d commandes lues, %d commandes chargées.", lues, chargees)
    motifs = Counter("doublon_client" if rejet[2].startswith("doublon_de_") else rejet[2] for rejet in rejets)
    for motif, nombre in motifs.items():
        journal.warning("%d lignes rejetées, motif %s.", nombre, motif)


@flow
def pipeline_etl_journal(fichier="commandes.csv"):
    # Étape 1 : les trois tâches de la partie B, puis journaliser
    clients, commandes, produits = extraire(fichier)
    clients_ok, commandes_ok, rejets, nb_dup = transformer(clients, commandes, produits)
    charger(clients_ok, produits, commandes_ok, rejets, fichier, len(commandes), nb_dup)
    journaliser(len(commandes), len(commandes_ok), rejets)


# Lancement du flux sur le fichier des commandes
pipeline_etl_journal("commandes.csv")

11:52:27.889 | INFO    | Flow run 'chocolate-civet' - Beginning flow run 'chocolate-civet' for flow 'pipeline-etl-journal'

11:52:27.902 | INFO    | Task run 'extraire-f03' - Finished in state Completed()

11:52:27.955 | INFO    | Task run 'transformer-685' - Finished in state Completed()

11:52:27.983 | INFO    | Task run 'charger-b6e' - Finished in state Completed()

11:52:27.994 | INFO    | Task run 'journaliser-026' - 2035 commandes lues, 1879 commandes chargées.

11:52:27.995 | WARNING | Task run 'journaliser-026' - 7 lignes rejetées, motif doublon_client.

... (4 outputs left out)

11:52:28.002 | WARNING | Task run 'journaliser-026' - 19 lignes rejetées, motif produit_inconnu.

11:52:28.004 | WARNING | Task run 'journaliser-026' - 39 lignes rejetées, motif montant_incoherent.

11:52:28.008 | INFO    | Task run 'journaliser-026' - Finished in state Completed()

11:52:28.892 | INFO    | Flow run 'chocolate-civet' - Finished in state Completed()

**Lecture :** dans les messages, la ligne « INFO » donne 2035 commandes lues et 1879 chargées.
Chaque ligne « WARNING » donne un motif de rejet et son nombre de lignes. Le flux a été relancé sur les mêmes fichiers : la base ne change pas, car le chargement est idempotent (voir S42).

## Partie D : Pannes et relances
Voir le cours : « Vocabulaire (5/6) », relance.

### Démo : D.1 une source en panne, sans relance
La fonction `outils.source_instable()` simule une source qui échoue deux fois, puis répond.

In [8]:
# Cette cellule lance un flux sans relance. Il n'y a rien à compléter.

@task
def appeler_source():
    return outils.source_instable()      # cette source échoue deux fois, puis répond


@flow
def flux_sans_relance():
    return appeler_source()


outils.reinitialiser_source_instable()   # on remet le compteur d'essais à zéro

# return_state=True : on récupère l'état du flux, au lieu de lever l'erreur
etat = flux_sans_relance(return_state=True)
print("Le flux a fini à l'état :", etat.name)

11:52:29.083 | INFO    | Flow run 'academic-koala' - Beginning flow run 'academic-koala' for flow 'flux-sans-relance'

11:52:29.089 | ERROR   | Task run 'appeler_source-0fd' - Task run failed with exception: ConnectionError('La source est indisponible (essai 1).')
Traceback (most recent call last):
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1058, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1755, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1075, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_2980\1910797387.py", line 5, in appeler_source
    return outils.source_instable()      # cette source échoue deux fois, puis répond
           ~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\data5\S45_Automatisation\TP2_Prefect\outils.py", line 52, in source_instable
    raise ConnectionError(f"La source est indisponible (essai {essai}).")
ConnectionError: La source est indisponible (essai 1).

11:52:29.097 | ERROR   | Task run 'appeler_source-0fd' - Finished in state Failed('Task run encountered an exception ConnectionError: La source est indisponible (essai 1).')

11:52:30.087 | ERROR   | Flow run 'academic-koala' - Encountered exception during execution: ConnectionError('La source est indisponible (essai 1).')
Traceback (most recent call last):
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1339, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 2133, in run_flow_sync
    engine.call_flow_fn()
    ~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1361, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_2980\1910797387.py", line 10, in flux_sans_relance
    return appeler_source()
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\tasks.py", line 1240, in __call__
    return run_task(
        task=self,
    ...<2 lines>...
        return_type=return_type,
    )
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1988, in run_task
    return run_task_sync(**kwargs)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1758, in run_task_sync
    return engine.state if return_type == "state" else engine.result()
                                                       ~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 622, in result
    raise self._raised
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1058, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1755, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1075, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_2980\1910797387.py", line 5, in appeler_source
    return outils.source_instable()      # cette source échoue deux fois, puis répond
           ~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\data5\S45_Automatisation\TP2_Prefect\outils.py", line 52, in source_instable
    raise ConnectionError(f"La source est indisponible (essai {essai}).")
ConnectionError: La source est indisponible (essai 1).

11:52:30.151 | INFO    | Flow run 'academic-koala' - Finished in state Failed('Flow run encountered an exception: ConnectionError: La source est indisponible (essai 1).')

Le flux a fini à l'état : Failed


**Lecture :** le flux a échoué dès le premier essai. Prefect affiche l'erreur en rouge : c'est normal.

### À vous : D.2 relancer automatiquement
**À faire :** décorez la tâche pour qu'elle soit relancée 3 fois, avec une seconde d'attente entre deux essais.

Indice : les options `retries` et `retry_delay_seconds` de `@task`.

In [9]:
# Objectif : relancer automatiquement la tâche qui échoue.
# Résultat attendu : le flux réussit au troisième essai.

# Étape 1 : le décorateur de la tâche, avec 3 relances et 1 seconde d'attente
@task(retries=3, retry_delay_seconds=1)
def appeler_source_robuste():
    return outils.source_instable()


@flow
def flux_avec_relances():
    return appeler_source_robuste()


outils.reinitialiser_source_instable()
print(flux_avec_relances())

11:52:30.550 | INFO    | Flow run 'vegan-catfish' - Beginning flow run 'vegan-catfish' for flow 'flux-avec-relances'

11:52:30.559 | INFO    | Task run 'appeler_source_robuste-e54' - Task run failed with exception: ConnectionError('La source est indisponible (essai 1).') - Retry 1/3 will start 1 second(s) from now

11:52:31.563 | INFO    | Task run 'appeler_source_robuste-e54' - Task run failed with exception: ConnectionError('La source est indisponible (essai 2).') - Retry 2/3 will start 1 second(s) from now

11:52:32.569 | INFO    | Task run 'appeler_source_robuste-e54' - Finished in state Completed()

11:52:33.555 | INFO    | Flow run 'vegan-catfish' - Finished in state Completed()

La source répond (essai 3).


**Lecture :** la source échoue aux essais 1 et 2. Prefect relance la tâche deux fois, et elle réussit au 3e essai.
Le flux finit à l'état « Completed ».

**Question :** une relance aide-t-elle quand le fichier source est abîmé ? Pourquoi ?

*Votre réponse :* non. L'erreur vient des données : relancer donne la même erreur. Une relance sert pour une panne passagère (réseau, serveur).

## Partie E : Contrôler le fichier
### À vous : E.1 arrêter le flux si le fichier est abîmé
Le flux de la partie B plante au milieu si le fichier est abîmé. Mieux vaut le contrôler avant.

**À faire :**
1. Écrivez la tâche `controler_source(fichier)` : elle contrôle `data/<fichier>` avec `outils.verifier_source(chemin, outils.COLONNES_COMMANDES)`. S'il y a des problèmes, elle lève une erreur `ValueError` qui les décrit.
2. Écrivez le flux `pipeline_securise(fichier)` : il contrôle le fichier, puis lance `pipeline_etl(fichier)` (le flux de la partie B).

In [10]:
# Objectif : arrêter le flux quand le fichier source est abîmé.
# Résultat attendu : le flux réussit pour commandes_increment.csv et échoue pour inexistant.csv.

@task
def controler_source(fichier):
    # Étape 1 : contrôler le fichier ; s'il y a des problèmes, lever une erreur ValueError qui les décrit
    problemes = outils.verifier_source("data/" + fichier, outils.COLONNES_COMMANDES)
    if problemes:
        raise ValueError("Fichier refusé : " + ", ".join(problemes))


@flow
def pipeline_securise(fichier):
    # Étape 1 : contrôler le fichier, puis lancer le flux pipeline_etl
    controler_source(fichier)
    pipeline_etl(fichier)


etat_correct = pipeline_securise("commandes_increment.csv", return_state=True)
etat_absent = pipeline_securise("inexistant.csv", return_state=True)

# Contrôle : l'état de chaque flux
print("Pour commandes_increment.csv, le flux a fini à l'état :", etat_correct.name)
print("Pour inexistant.csv, le flux a fini à l'état :", etat_absent.name)

11:52:33.780 | INFO    | Flow run 'sociable-bobcat' - Beginning flow run 'sociable-bobcat' for flow 'pipeline-securise'

11:52:33.788 | INFO    | Task run 'controler_source-4a6' - Finished in state Completed()

11:52:34.042 | INFO    | Flow run 'shrewd-lynx' - Beginning subflow run 'shrewd-lynx' for flow 'pipeline-etl'

11:52:34.052 | INFO    | Task run 'extraire-e48' - Finished in state Completed()

11:52:34.074 | INFO    | Task run 'transformer-6a0' - Finished in state Completed()

11:52:34.091 | INFO    | Task run 'charger-fd2' - Finished in state Completed()

... (4 outputs left out)

11:52:36.011 | ERROR   | Task run 'controler_source-c0c' - Finished in state Failed('Task run encountered an exception ValueError: Fichier refusé : fichier introuvable')

11:52:37.000 | ERROR   | Flow run 'silky-chinchilla' - Encountered exception during execution: ValueError('Fichier refusé : fichier introuvable')
Traceback (most recent call last):
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1339, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 2133, in run_flow_sync
    engine.call_flow_fn()
    ~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\flow_engine.py", line 1361, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_2980\1170952086.py", line 15, in pipeline_securise
    controler_source(fichier)
    ~~~~~~~~~~~~~~~~^^^^^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\tasks.py", line 1240, in __call__
    return run_task(
        task=self,
    ...<2 lines>...
        return_type=return_type,
    )
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1988, in run_task
    return run_task_sync(**kwargs)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1758, in run_task_sync
    return engine.state if return_type == "state" else engine.result()
                                                       ~~~~~~~~~~~~~^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 622, in result
    raise self._raised
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1058, in run_context
    yield self
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1755, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\task_engine.py", line 1075, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\msakkari\Documents\IUT 2026_2027\data5_26_27\.venv\Lib\site-packages\prefect\utilities\callables\__init__.py", line 404, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\msakkari\AppData\Local\Temp\ipykernel_2980\1170952086.py", line 9, in controler_source
    raise ValueError("Fichier refusé : " + ", ".join(problemes))
ValueError: Fichier refusé : fichier introuvable

11:52:37.023 | INFO    | Flow run 'silky-chinchilla' - Finished in state Failed('Flow run encountered an exception: ValueError: Fichier refusé : fichier introuvable')

Pour commandes_increment.csv, le flux a fini à l'état : Completed
Pour inexistant.csv, le flux a fini à l'état : Failed


**Lecture :** le flux du fichier absent est « Failed » : la tâche `controler_source` a levé une erreur, et le pipeline n'a pas démarré.
Le message d'erreur en rouge est normal.

## Partie F : Planifier
Voir le cours : « Vocabulaire (4/6) », planification.

### À vous : F.1 lancer le pipeline toutes les 30 secondes
Un flux planifié doit tourner en permanence. On ne peut pas le lancer depuis le notebook : on écrit un script, qu'on lance dans un terminal.

**À faire :** complétez le script `serveur_prefect.py`.
1. La tâche `lancer_pipeline(fichier)` ouvre la base `etl.DB`, appelle `etl.pipeline(con, fichier, flow_run.name, declencheur="prefect")`, puis ferme la base.
2. Le flux `pipeline_planifie()` lance cette tâche sur `commandes_increment.csv`.
3. Pour finir, on publie le flux : il se lance tout seul toutes les 30 secondes.

Indice : la méthode `.serve()` d'un flux, avec les options `name` et `interval`.

In [25]:
%%writefile serveur_prefect.py
# Objectif : lancer le pipeline automatiquement, à intervalle régulier.
# Résultat attendu : dans l'interface de Prefect, une nouvelle exécution apparaît toutes les 30 secondes.
import sqlite3

from prefect import flow, task
from prefect.runtime import flow_run

import etl


@task
def lancer_pipeline(fichier):
    # Étape 1 : ouvrir la base, lancer le pipeline sur le fichier, fermer la base
    con = sqlite3.connect(etl.DB)
    etl.pipeline(con, fichier, flow_run.name, declencheur="prefect")
    con.close()


@flow
def pipeline_planifie():
    # Étape 2 : lancer la tâche sur commandes_increment.csv
    lancer_pipeline("commandes_increment.csv")


if __name__ == "__main__":
    # Étape 3 : publier le flux, avec un nom et un intervalle de 30 secondes
    pipeline_planifie.serve(name="pipeline-toutes-les-30-secondes", interval=30)


Overwriting serveur_prefect.py


### Démo : F.2 noter le nombre d'exécutions avant le lancement
Des exécutions lancées par Prefect existent déjà (parties B à E). Cette cellule note leur nombre, pour pouvoir le comparer après.

In [26]:
# Cette cellule note le nombre d'exécutions déjà enregistrées. Il n'y a rien à compléter.
con = sqlite3.connect(etl.DB)
avant_serveur = con.execute("SELECT COUNT(*) FROM etl_log WHERE declencheur = 'prefect'").fetchone()[0]
con.close()
print("Avant le lancement, le journal etl_log contient", avant_serveur, "exécutions lancées par Prefect.")

Avant le lancement, le journal etl_log contient 28 exécutions lancées par Prefect.


### À vous : F.3 lancer le script
**Pour lancer le script :**
1. Ouvrez un premier terminal, dans le dossier du TP. Tapez `prefect server start` et laissez-le ouvert.
   L'interface de Prefect est alors sur http://127.0.0.1:4200.
2. Ouvrez un second terminal, dans le dossier du TP. Tapez une seule fois : `prefect config set PREFECT_API_URL=http://127.0.0.1:4200/api`
3. Dans ce second terminal, tapez : `python serveur_prefect.py` (laissez-le ouvert)
4. Attendez deux minutes. Dans l'interface, regardez la page « Runs ».
5. Arrêtez les deux terminaux avec `Ctrl+C`.

**Question :** combien d'exécutions « Completed » voyez-vous après deux minutes ? Pourquoi ce nombre ?

*Votre réponse :* environ 4 : une exécution toutes les 30 secondes. Les exécutions à venir apparaissent à l'état « Scheduled ».

### Démo : F.4 compter les nouvelles exécutions
Après avoir arrêté les deux terminaux, exécutez la cellule suivante.

In [13]:
# Cette cellule compte les exécutions lancées par le script. Il n'y a rien à compléter.
con = sqlite3.connect(etl.DB)
apres_serveur = con.execute("SELECT COUNT(*) FROM etl_log WHERE declencheur = 'prefect'").fetchone()[0]
con.close()
print("Le script a lancé le pipeline", apres_serveur - avant_serveur, "fois.")

Le script a lancé le pipeline 0 fois.


## Partie G : Lancer quand un fichier change
Voir le cours : « Vocabulaire (3/6) », déclencheur.

### À vous : G.1 une surveillance avec Prefect
En S42, un script surveillait un fichier en boucle. Ici, Prefect fait le travail : le flux se lance toutes les 10 secondes, et ne lance le pipeline que si le fichier a changé.

**À faire :** complétez le script `surveillance_prefect.py`. La tâche `lancer_pipeline` est donnée.
1. La tâche `fichier_a_change()` :
   - calcule l'empreinte actuelle du fichier avec `outils.empreinte(FICHIER)` ;
   - la compare à celle enregistrée dans le fichier `MEMOIRE` (s'il n'existe pas, le fichier est considéré comme changé) ;
   - enregistre l'empreinte actuelle dans `MEMOIRE` ;
   - renvoie `True` si le fichier a changé, sinon `False`.
2. Le flux `surveiller()` lance `lancer_pipeline()` si le fichier a changé. Sinon, il écrit dans le journal que le fichier n'a pas changé.

Indices : `json.load` et `json.dump` pour lire et écrire `MEMOIRE`.

In [14]:
%%writefile surveillance_prefect.py
# Objectif : lancer le pipeline seulement quand le fichier a changé.
# Résultat attendu : le pipeline se lance au premier passage et après chaque modification du fichier, jamais quand le fichier est identique.
import json
import os
import sqlite3

from prefect import flow, get_run_logger, task
from prefect.runtime import flow_run

import etl
import outils

FICHIER = "data/commandes_increment.csv"
MEMOIRE = "derniere_empreinte.json"     # on y garde l'empreinte vue au dernier passage


@task
def fichier_a_change():
    # Étape 1 : l'empreinte actuelle du fichier
    # Étape 2 : l'empreinte enregistrée au dernier passage (aucune si MEMOIRE n'existe pas)
    # Étape 3 : enregistrer l'empreinte actuelle, puis renvoyer True si le fichier a changé
    actuelle = outils.empreinte(FICHIER)
    precedente = None
    if os.path.exists(MEMOIRE):
        precedente = json.load(open(MEMOIRE, encoding="utf8"))["empreinte"]
    json.dump({"empreinte": actuelle}, open(MEMOIRE, "w", encoding="utf8"))
    return actuelle != precedente


@task
def lancer_pipeline():
    con = sqlite3.connect(etl.DB)
    etl.pipeline(con, os.path.basename(FICHIER), flow_run.name, declencheur="prefect")
    con.close()
    get_run_logger().info("Le pipeline a été lancé sur %s.", FICHIER)


@flow
def surveiller():
    # Étape 4 : lancer le pipeline si le fichier a changé, sinon écrire un message dans le journal
    if fichier_a_change():
        lancer_pipeline()
    else:
        get_run_logger().info("Le fichier n'a pas changé : le pipeline n'est pas lancé.")


if __name__ == "__main__":
    surveiller.serve(name="surveillance", interval=10)


Writing surveillance_prefect.py


### Démo : G.2 tester la surveillance, sans serveur
La cellule suivante lance le flux trois fois. Entre le deuxième et le troisième passage, une commande est ajoutée au fichier.
Lisez les messages de Prefect.

In [28]:
# Cette cellule teste le flux de surveillance. Il n'y a rien à compléter.
import importlib

import surveillance_prefect
importlib.reload(surveillance_prefect)     # recharge le script que vous venez d'écrire
from surveillance_prefect import surveiller

if os.path.exists("derniere_empreinte.json"):
    os.remove("derniere_empreinte.json")   # on repart sans mémoire

print("--- Premier passage : aucune empreinte enregistrée ---")
surveiller()

print("--- Deuxième passage : le fichier n'a pas changé ---")
surveiller()

nouvelle = outils.ajouter_commande_valide()
print("--- Troisième passage : la commande", nouvelle, "vient d'être ajoutée au fichier ---")
surveiller()

# Contrôle : nombre de commandes dans la base
con = sqlite3.connect(etl.DB)
print("La table commande contient", con.execute("SELECT COUNT(*) FROM commande").fetchone()[0], "lignes.")
con.close()

--- Premier passage : aucune empreinte enregistrée ---


12:40:42.653 | INFO    | Flow run 'evasive-mongrel' - Beginning flow run 'evasive-mongrel' for flow 'surveiller'

12:40:42.662 | INFO    | Task run 'fichier_a_change-8bc' - Finished in state Completed()

12:40:42.726 | INFO    | Task run 'lancer_pipeline-c9c' - Le pipeline a été lancé sur data/commandes_increment.csv.

12:40:42.729 | INFO    | Task run 'lancer_pipeline-c9c' - Finished in state Completed()

12:40:43.656 | INFO    | Flow run 'evasive-mongrel' - Finished in state Completed()

... (8 outputs left out)

12:40:45.189 | INFO    | Task run 'lancer_pipeline-d8f' - Le pipeline a été lancé sur data/commandes_increment.csv.

12:40:45.194 | INFO    | Task run 'lancer_pipeline-d8f' - Finished in state Completed()

12:40:46.113 | INFO    | Flow run 'vagabond-alligator' - Finished in state Completed()

La table commande contient 1948 lignes.


**Lecture :** le pipeline s'est lancé au premier passage et au troisième, mais pas au deuxième.
La commande ajoutée est dans la base : la table `commande` contient 1940 lignes (si vous exécutez la démo une seule fois : chaque exécution ajoute une commande au fichier).

**Pour la lancer en continu :** dans un terminal, dans le dossier du TP, tapez `python surveillance_prefect.py` (le serveur Prefect de la partie F doit tourner).
Ajoutez ensuite une commande depuis un autre terminal ou depuis le notebook, et regardez l'interface de Prefect.

### Démo : G.3 modifier une valeur et observer
La commande ajoutée en G.2 est dans la variable `nouvelle`. On modifie sa quantité à la main, puis on regarde la base.

1. Exécutez la première cellule : elle affiche la commande avant la modification.
2. Ouvrez `data/commandes_increment.csv`. Retrouvez la ligne de cette commande. Changez sa quantité, puis enregistrez le fichier.
3. Exécutez la deuxième cellule : le flux détecte le changement et lance le pipeline.
4. Ré-exécutez la première cellule : la ligne affiche la nouvelle valeur.

Si la nouvelle valeur est invalide (une quantité négative, par exemple), la ligne est rejetée. Exécutez alors la troisième cellule pour lire le rejet.

In [22]:
# Cette cellule affiche la commande ajoutée en G.2. Il n'y a rien à compléter.
con = sqlite3.connect(etl.DB)
print(pd.read_sql("SELECT * FROM commande WHERE id_commande = ?", con, params=(nouvelle,)))
con.close()

   id_commande  id_client   sku  quantite date_commande  montant_ttc  statut
0         2067          1  P001         2    2026-10-01        20.05  livree


In [23]:
# Cette cellule lance un passage de la surveillance. Il n'y a rien à compléter.
surveiller()

12:12:42.985 | INFO    | Flow run 'vivid-kudu' - Beginning flow run 'vivid-kudu' for flow 'surveiller'

12:12:42.997 | INFO    | Task run 'fichier_a_change-569' - Finished in state Completed()

12:12:42.999 | INFO    | Flow run 'vivid-kudu' - Le fichier n'a pas changé : le pipeline n'est pas lancé.

12:12:43.989 | INFO    | Flow run 'vivid-kudu' - Finished in state Completed()

In [ ]:
# Cette cellule affiche les rejets de la commande ajoutée en G.2. Il n'y a rien à compléter.
con = sqlite3.connect(etl.DB)
print(pd.read_sql("SELECT source, cle, motif, date_rejet FROM rejet WHERE cle = ?", con, params=(nouvelle,)))
con.close()

## Bilan
Répondez en une ligne chacune.
1. Qu'apporte Prefect, par rapport au script de surveillance du TP S42 ?
2. Quelle est la différence entre un flux et une tâche ?
3. Quand une relance est-elle utile ?
4. Quel déclencheur avez-vous utilisé en partie F, et lequel en partie G ?
5. Que manque-t-il encore pour qu'un pipeline soit prêt pour la production ?

*Votre réponse :* (réponses libres)